# C12-classical-models — Practice p29 — Solution


Both rounds consume the normalized weights from the immediately preceding round; the returned ledger retains every scalar and array named by the calculation.


In [ ]:
import numpy as np

t_p29 = np.array([-1, -1, 1, 1], dtype=np.int64)
h1_p29 = np.array([-1, 1, 1, 1], dtype=np.int64)
h2_p29 = np.array([-1, -1, -1, 1], dtype=np.int64)


def adaboost_two_rounds(t, h1, h2):
    arrays=[np.asarray(value) for value in (t,h1,h2)]
    if any(value.ndim!=1 for value in arrays) or not (arrays[0].shape==arrays[1].shape==arrays[2].shape) or arrays[0].size==0:
        raise ValueError("matching nonempty vectors required")
    if arrays[0].size!=4 or any(not np.all((value==-1)|(value==1)) for value in arrays):
        raise ValueError("four sign entries required")
    t,h1,h2=[value.astype(np.float64) for value in arrays]
    q1=np.full(4,0.25,dtype=np.float64)
    error1=float(q1[t!=h1].sum()); alpha1=float(0.5*np.log((1-error1)/error1))
    unnormalized_q2=q1*np.exp(-alpha1*t*h1); Z1=float(unnormalized_q2.sum())
    # PLAN018_MUTATION_TARGET: C12-p29-weight-update
    q2=unnormalized_q2/Z1
    error2=float(q2[t!=h2].sum()); alpha2=float(0.5*np.log((1-error2)/error2))
    unnormalized_q3=q2*np.exp(-alpha2*t*h2); Z2=float(unnormalized_q3.sum()); q3=unnormalized_q3/Z2
    scores=(alpha1*h1+alpha2*h2).astype(np.float64)
    predictions=np.where(scores>=0.0,1,-1).astype(np.int64)
    return {"q1":q1,"unnormalized_q2":unnormalized_q2.astype(np.float64),"q2":q2.astype(np.float64),
            "unnormalized_q3":unnormalized_q3.astype(np.float64),"q3":q3.astype(np.float64),
            "error1":error1,"alpha1":alpha1,"Z1":Z1,"error2":error2,"alpha2":alpha2,"Z2":Z2,
            "scores":scores,"predictions":predictions}


ledger_p29 = adaboost_two_rounds(t_p29, h1_p29, h2_p29)
symbolic_ledger_p29 = "Round 1: error=1/4, alpha=(1/2)log 3, unnormalized=(1/(4sqrt3),sqrt3/4,1/(4sqrt3),1/(4sqrt3)), Z=sqrt3/2, q2=(1/6,1/2,1/6,1/6). Round 2: error=1/6, alpha=(1/2)log 5, unnormalized=(1/(6sqrt5),1/(2sqrt5),sqrt5/6,1/(6sqrt5)), Z=sqrt5/3, q3=(1/10,3/10,1/2,1/10)."
interpretation_p29 = '''Uniform round-2 weights would give error 1/4 and alpha=(1/2)log 3, discarding the first learner's emphasis and therefore breaking sequential correction.'''


### Answer check


In [ ]:
ATOL=1e-12
RTOL=1e-10
assert set(ledger_p29)=={"q1","unnormalized_q2","q2","unnormalized_q3","q3","error1","alpha1","Z1","error2","alpha2","Z2","scores","predictions"}
assert np.isclose(ledger_p29["error1"],0.25,atol=ATOL,rtol=RTOL)
assert np.isclose(ledger_p29["alpha1"],0.5*np.log(3),atol=ATOL,rtol=RTOL)
# PLAN018_ANSWER_CHECK: C12-p29-adaboost-ledger
assert np.allclose(ledger_p29["q2"],[1/6,1/2,1/6,1/6],atol=ATOL,rtol=RTOL)
assert np.isclose(ledger_p29["error2"],1/6,atol=ATOL,rtol=RTOL)
assert np.isclose(ledger_p29["alpha2"],0.5*np.log(5),atol=ATOL,rtol=RTOL)
assert np.allclose(ledger_p29["q3"],[0.1,0.3,0.5,0.1],atol=ATOL,rtol=RTOL)
assert np.array_equal(ledger_p29["predictions"],[-1,-1,-1,1])
assert "log 5" in symbolic_ledger_p29 and "Uniform" in interpretation_p29
